In [1]:
from mpi4py import MPI  # must be first: pre-init MPI before kernel threads start
#@title Define functions

import numpy as np
import matplotlib
matplotlib.use("Agg")  # headless backend for Docker
import matplotlib.pyplot as plt
from collections import deque
import dolfin as df
import sys
from graphnics import *
from math import gamma
from argparse import RawDescriptionHelpFormatter
from petsc4py import PETSc
from xii import *
import time
import networkx as nx
from scipy.spatial import cKDTree
from itertools import combinations
from collections import deque

import os

def load_patient_vtp(vtp_path):
    """
    Load pt002_inflow_centerline.vtp into a FenicsGraph.
    The raw network has 32 cycles; a maximum-spanning-tree is extracted,
    weighted by pressure gradient for real edges vs near-zero for phantom edges.
    Returns (FenicsGraph tree, root) where root is the inlet vertex (highest pressure).
    """
    import vtk
    reader = vtk.vtkXMLPolyDataReader()
    reader.SetFileName(vtp_path)
    reader.Update()
    poly = reader.GetOutput()

    n_pts = poly.GetNumberOfPoints()
    pts   = np.array([poly.GetPoint(i) for i in range(n_pts)])

    pd       = poly.GetPointData()
    pressure = np.array([pd.GetArray('pressure_mmhg').GetValue(i) for i in range(n_pts)])
    radius_p = np.array([pd.GetArray('radius_mm').GetValue(i)     for i in range(n_pts)])

    cd             = poly.GetCellData()
    is_phantom_arr = cd.GetArray('is_phantom')

    G_nx = nx.Graph()
    for i in range(n_pts):
        G_nx.add_node(i, pos=pts[i].tolist(),
                      radius=float(radius_p[i]),
                      pressure=float(pressure[i]))

    lines   = poly.GetLines()
    id_list = vtk.vtkIdList()
    lines.InitTraversal()
    cell_idx = 0
    while lines.GetNextCell(id_list):
        n_ids   = id_list.GetNumberOfIds()
        phantom = bool(is_phantom_arr.GetValue(cell_idx)) if is_phantom_arr else False
        prev    = int(id_list.GetId(0))
        for k in range(1, n_ids):
            curr   = int(id_list.GetId(k))
            length = float(np.linalg.norm(pts[curr] - pts[prev]))
            dp     = abs(float(pressure[curr]) - float(pressure[prev]))
            weight = (dp / (length + 1e-10)) if not phantom else 1e-6
            if not G_nx.has_edge(prev, curr):
                G_nx.add_edge(prev, curr, is_phantom=phantom, length=length, weight=weight)
            prev = curr
        cell_idx += 1

    n_comp = nx.number_connected_components(G_nx)
    print(f"Raw graph: {G_nx.number_of_nodes()} nodes, {G_nx.number_of_edges()} edges, "
          f"{n_comp} component(s)")

    T = nx.maximum_spanning_tree(G_nx, weight='weight')
    phantom_kept = sum(1 for u,v,d in T.edges(data=True) if d.get('is_phantom', False))
    print(f"Spanning tree: {T.number_of_nodes()} nodes, {T.number_of_edges()} edges "
          f"({phantom_kept} phantom edges retained for connectivity)")

    inlet = int(np.argmax(pressure))
    print(f"Inlet vertex: {inlet},  p = {pressure[inlet]:.1f} mmHg,  "
          f"pos = {[round(x,2) for x in pts[inlet].tolist()]}")

    # FenicsGraph is a DiGraph: every edge must point away from the root (parent -> child),
    # otherwise downstream BFS/tangent/mesh logic only sees a few directly-oriented edges.
    # nx.maximum_spanning_tree returns an undirected tree with arbitrary edge orientation,
    # so re-orient via BFS from the inlet before building FG.
    T_dir = nx.bfs_tree(T, source=inlet)
    n_reachable = T_dir.number_of_nodes()
    if n_reachable != T.number_of_nodes():
        print(f"WARNING: only {n_reachable}/{T.number_of_nodes()} nodes reachable from inlet in spanning tree")

    FG = FenicsGraph()
    for n, d in sorted(T.nodes(data=True)):
        FG.add_node(n, pos=d['pos'], radius=d['radius'])
    for u, v in T_dir.edges():
        edata = T.edges[u, v]
        r = 0.5 * (G_nx.nodes[u]['radius'] + G_nx.nodes[v]['radius'])
        FG.add_edge(u, v, radius=r, is_phantom=edata.get('is_phantom', False))

    return FG, inlet

# === Load patient centerline (commented out for Y-bif diagnostic) ===
# VTP_PATH = "pt002_inflow_centerline.vtp"
# G, root  = load_patient_vtp(VTP_PATH)
# junctions = [n for n in G.nodes if G.degree[n] > 2]
# terminals = [n for n in G.nodes if G.degree[n] == 1]
# print(f"#nodes: {len(G.nodes)}  #edges: {len(G.edges)}")
# print(f"#junctions: {len(junctions)}  #terminals: {len(terminals)}")

def assign_vessel_ids_by_junction_terminal_paths(G, min_edge_length=1e-6):
    """
    Vessel = maximal path between important nodes:
      important = terminals (deg=1) and junctions (deg>=3)

    Also ignores ultra-short edges (from duplicated points / merge artifacts).
    """
    pos = nx.get_node_attributes(G, "pos")

    def elen(u, v):
        pu = np.array(pos[u], float); pv = np.array(pos[v], float)
        return float(np.linalg.norm(pu - pv))

    important = {n for n in G.nodes if (G.degree[n] == 1 or G.degree[n] >= 3)}

    used = set()
    vessel_id = 0

    def ekey(u, v):  # undirected key
        return (u, v) if u <= v else (v, u)

    for s in important:
        for nb in G.neighbors(s):
            if elen(s, nb) < min_edge_length:
                continue

            e0 = ekey(s, nb)
            if e0 in used:
                continue

            # walk from s -> nb until next important node
            path = [s, nb]
            prev, cur = s, nb
            used.add(e0)

            while cur not in important:
                nbs = [x for x in G.neighbors(cur) if elen(cur, x) >= min_edge_length]
                if len(nbs) == 0:
                    break
                if len(nbs) == 1:
                    nxt = nbs[0]
                else:
                    # choose neighbor that's not prev; if multiple, pick the one with smallest turning angle (straightest)
                    candidates = [x for x in nbs if x != prev]
                    if not candidates:
                        break
                    # pick the "straightest" continuation
                    pc = np.array(pos[cur], float)
                    pprev = np.array(pos[prev], float)
                    v_in = pc - pprev
                    v_in /= (np.linalg.norm(v_in) + 1e-16)

                    best = None
                    best_dot = -1e9
                    for x in candidates:
                        px = np.array(pos[x], float)
                        v_out = px - pc
                        v_out /= (np.linalg.norm(v_out) + 1e-16)
                        d = float(np.dot(v_in, v_out))
                        if d > best_dot:
                            best_dot = d
                            best = x
                    nxt = best

                e = ekey(cur, nxt)
                if e in used:
                    break

                used.add(e)
                path.append(nxt)
                prev, cur = cur, nxt

            # label all edges in this path with vessel_id
            for i in range(len(path) - 1):
                u, v = path[i], path[i+1]
                G.edges[u, v]["vessel_id"] = vessel_id

            vessel_id += 1

    return vessel_id

# nv = assign_vessel_ids_by_junction_terminal_paths(G, min_edge_length=1e-5)  # runs in Cell 2 for Y-bif

def bif_tangents(G):
    pos   = nx.get_node_attributes(G,"pos")
    nodes = list(G.nodes)
    T     = np.zeros((len(nodes),3))
    for k, n in enumerate(nodes):
        p0 = np.array(pos[n])
        neighbor = list(G.neighbors(n))
        vec = np.zeros(3)
        for m in neighbor:
            vec += np.array(pos[m]) - p0

        norm = np.linalg.norm(vec)
        if norm > 0:
          T[k,:] = vec/norm
        else:
          T[k,:] = np.array([0,0,1.0]) #use unit normal if needed
    P = np.array([pos[n] for n in nodes])
    return nodes, P, T

class TangentExpression(UserExpression):
    def __init__(self, nodes, P, T_array, **kwargs):
        self.nodes  = nodes
        self.P      = np.asarray(P)
        self.T_array = np.asarray(T_array)
        self.kdtree = cKDTree(self.P)
        super().__init__(**kwargs)
    def eval(self,values,x):
      _, idx = self.kdtree.query((x[0],x[1],x[2]))
      tx, ty, tz = self.T_array[idx]
      values[0] = tx
      values[1] = ty
      values[2] = tz
    def value_shape(self):
      return (3,)

def build_bifurcation_pairs_bifs(G, root, *, require_tree_like=True):
    """
    General parent/daughter classification at junctions using graph-distance from a chosen inlet/root.

    Parameters
    ----------
    G : networkx.Graph-like
        Works with FenicsGraph or nx.Graph. Must support neighbors(), degree, nodes data with "pos".
    root : int
        Upstream inlet node id in G.
    require_tree_like : bool
        If True, enforce that each junction has exactly one upstream neighbor.
        If False, allow ambiguous cases and pick parent by smallest distance.

    Returns
    -------
    results : list[dict]
        Each dict has:
          - bif_node
          - parent (graph node id)
          - daughters (list of graph node ids)
          - parent_id (always 1)
          - daughter_id (2..)
          - pairs: [(1,2),(1,3),(2,3)] etc.
          - pos: np.array([x,y,z])
          - neighbor_sorted: [parent] + daughters
          - vessel_id: {parent:1, daughter1:2, ...}
          - dist_b: distance(root -> bif_node)
          - dist_neighbors: {nb: dist(root -> nb)}
    """

    H = nx.Graph()
    H.add_nodes_from(G.nodes(data=True))
    H.add_edges_from(G.edges())

    pos  = nx.get_node_attributes(H, "pos")
    dist = dict(nx.single_source_shortest_path_length(H, source=root))

    bif_nodes = [n for n in H.nodes if H.degree[n] > 2]
    results = []

    for b in bif_nodes:
        nbrs = list(H.neighbors(b))
        db = dist.get(b, None)
        if db is None:
            if require_tree_like:
                raise RuntimeError(f"Junction {b} not reachable from root={root}.")
            continue

        # STRICT tree rule: parent is db-1, daughters are db+1
        upstream   = [n for n in nbrs if dist.get(n, None) == db - 1]
        downstream = [n for n in nbrs if dist.get(n, None) == db + 1]

        if require_tree_like and len(upstream) != 1:
            raise RuntimeError(
                f"Junction {b} ambiguous: dist[b]={db}, "
                f"upstream={upstream}, downstream={downstream}, neighbors={nbrs}"
            )

        parent = upstream[0] if len(upstream) else min(nbrs, key=lambda n: dist.get(n, 10**18))
        daughters = sorted([n for n in nbrs if n != parent],
                           key=lambda n: (dist.get(n, 10**18), int(n)))

        vessel_id = {parent: 1}
        for k, d in enumerate(daughters, start=2):
            vessel_id[d] = k

        daughter_id = [vessel_id[d] for d in daughters]
        pairs = [(1, did) for did in daughter_id] + list(combinations(daughter_id, 2))

        results.append({
            "bif_node": b,
            "parent": parent,
            "daughters": daughters,
            "pairs": pairs,
            "pos": np.array(pos[b], dtype=float),
            "neighbor_sorted": [parent] + daughters,
            "dist_b": db,
            "dist_neighbors": {n: dist.get(n, None) for n in nbrs},
        })

    results.sort(key=lambda r: (r["dist_b"], int(r["bif_node"])))
    return results

# root is returned by load_patient_vtp above (vertex 335 for pt002, p=882 mmHg)
# bif_info is computed inside the simulation cell below

def bifurcation_vertices_from_graph(Lambda, bif_info):
    """Return mesh vertex indices of bifurcation points (nearest vertex to each bif node position)."""
    coords = Lambda.coordinates()

    def nearest_vertex(point):
        diff = coords - point
        return int(np.argmin(np.linalg.norm(diff, axis=1)))

    bif_vertices = []
    for bif in bif_info:
        vtx = nearest_vertex(bif["pos"])
        bif_vertices.append(vtx)
    return sorted(set(bif_vertices))

def add_kirchhoff_junction_terms_P1DG(
    A_petsc, V3, V1, Lambda, bif_vertices, D_area_expr,
    eta_gamma=10.0, epsilon=1.0, diff_coef=1.0
):
    """
    Assemble Kirchhoff/SIPG junction terms by writing directly into the PETSc Mat.
    A_petsc - global stiffness matrix assembled by FEniCS for facet terms. This function will add to it
    V1 - DG1 function space
    Lambda - 1D mesh
    bif_vertices - list of vertex indices where 3 branches meet (bifurcation)
    eta_gamma - penalty
    epsilon - SIPG parameter
    """
    dm = V1.dofmap() # maps cell index to its DOF indices
    coords = Lambda.coordinates() # xyz coordinates of every vertex

    Lambda.init(0, 1) # build vertex-cell connectivity 
    v2c = Lambda.topology()(0, 1) # returns all cells touching that vertex

    N = A_petsc.mat().getSize()[0] # total DOFs in entire system (3D and 1D)
    off1 = N - V1.dim()   # finds where 1d is in 3d in the global matrix

    # get petsc4py Mat
    M = A_petsc.mat() #PETSc matrix object

    M.setOption(PETSc.Mat.Option.NEW_NONZERO_ALLOCATION_ERR, False)
    M.setOption(PETSc.Mat.Option.NEW_NONZERO_LOCATION_ERR, False)

    def add_entry(I, J, val):
        # addv=True means accumulate into existing entry
        M.setValue(I, J, val, addv=True)

    for vtx in bif_vertices:
        incident_cells = list(v2c(vtx)) # all branches meeting at current vertex
        m = len(incident_cells) # number of branches
        if m < 3: 
            continue

        branches = []
        h_gamma = 0.0

        # collect geomtric data for each branch at bifurcation
        # c is integer index of one branch that touches the vertex (vtx) so it runs 3 times for each bifurcation
        for c in incident_cells:
            cell = Cell(Lambda, c)
            vids = cell.entities(0)  # indices of 1D cells, 2 per cell
            x0 = coords[vids[0]]  # upstream cell index
            x1 = coords[vids[1]] # dwonstream cell index
            hK = float(np.linalg.norm(x1 - x0)) # length of cell
            h_gamma = max(h_gamma, hK) # penalty

            if vids[0] == vtx: # determining where the bifurcation is actually connecting two vessels
                loc_g, loc_o = 0, 1 # bifurcation connects at the left end of the cell
                x_mid = 0.5*(x0 + x1) # used later for cross-sectional area
            else:
                loc_g, loc_o = 1, 0 # bifurcation connects at the right end of the cell
                x_mid = 0.5*(x0 + x1)

            cdofs = dm.cell_dofs(c)  # DG1: 2 dofs per cell, ex) [83,91]
            dof_g = int(cdofs[loc_g]) # =cdofs[1] = 91 <- junction part of cell
            dof_o = int(cdofs[loc_o]) # =cdofs[0] = 83 <- far-end part of cell

            Da = float(D_area_expr(Point(*x_mid))) # cross-sectional area
            branches.append(dict(hK=hK, Da=Da, dof_g=dof_g, dof_o=dof_o)) 

        # branch pairings, runs for every unorderd pair (a,b) once
        # If bifurcation has parent vessel 1, daughters 2,3 then pairings are (1,2), (2,3), (1,3)
        # recall, m is number of branches (only bifurcations in this network so m = 3)
        for a in range(m):
            for b in range(a + 1, m):
                ha, Da = branches[a]["hK"], branches[a]["Da"] # length and area of branch a
                hb, Db = branches[b]["hK"], branches[b]["Da"] # length and area of branch b

                ga = branches[a]["dof_g"] # u1_a(v_j) - branch a value at junction
                oa = branches[a]["dof_o"] # u1_a(v_(j-k)) - branch a value at far end
                gb = branches[b]["dof_g"] # u1_b(v_j) - branch b value at junction
                ob = branches[b]["dof_o"] # u1_b(v_(j-k)) - branch b value at far end

                # GLOBAL indices
                Gga = off1 + ga   # global index of u1_a(v_j)
                Goa = off1 + oa   # global index of u1_a,far
                Ggb = off1 + gb   # global index of u1_b(v_j)
                Gob = off1 + ob   # global index of u1_b,far

                # === penalty ===
                # implements pen*(u1_a(v_j)-u1_b(v_j))*(v1_a(v_j)-v1_b(v_j)) (term 3 in manuscript)
                pen = eta_gamma * 0.5 * (Da + Db) / h_gamma

                # row v1_a, col u1_a  ->  +pen * u1_a * v1_a
                add_entry(Gga, Gga, +pen)

                # row v1_a, col u1_b  ->  -pen * u1_b * v1_a
                add_entry(Gga, Ggb, -pen)

                # row v1_b, col u1_a  ->  -pen * u1_a * v1_b
                add_entry(Ggb, Gga, -pen)

                # row v1_b, col u1_b  ->  +pen * u1_b * v1_b
                add_entry(Ggb, Ggb, +pen)

                w = -diff_coef / m
                # === Consistency ===
                # implements -(1/m)*Da*du1_a/ds - Db*du1_b/ds)*(v1_a(v_j)-v1_b(v_j)) (Term 1 in manuscript)
                # Test fn v1_a (row = Gga)
                # Da*du1_a/ds term:  +Da/ha on u1_a,far,  -Da/ha on u1_a(v_j)
                add_entry(Gga, Goa, w * (+Da/ha))   # row v1_a, col u1_a,far
                add_entry(Gga, Gga, w * (-Da/ha))   # row v1_a, col u1_a(v_j)

                # Db*du1_b/ds term (negated because it's Da*flux_a minus Db*flux_b):
                add_entry(Gga, Gob, w * (-Db/hb))   # row v1_a, col u1_b,far
                add_entry(Gga, Ggb, w * (+Db/hb))   # row v1_a, col u1_b(v_j)

                # Test fn v1_b (row = Ggb) — everything flips sign from previous four lines
                add_entry(Ggb, Goa, w * (-Da/ha))   # row v1_b, col u1_a,far
                add_entry(Ggb, Gga, w * (+Da/ha))   # row v1_b, col u1_a(v_j)
                add_entry(Ggb, Gob, w * (+Db/hb))   # row v1_b, col u1_b,far
                add_entry(Ggb, Ggb, w * (-Db/hb))   # row v1_b, col u1_b(v_j)

                # implements -(epsilon/m) * (Da·dv1_a/ds - Db·∂d1_b/ds) * (u1_a(v_j) - u1_b(v_j)) (term 2 in manuscript)
                ws = -diff_coef * epsilon / m

                # Trial fn u1_a (col = Gga) 
                add_entry(Goa, Gga, ws * (+Da/ha))   # row u1_a,far,   col u1_a(v_j)
                add_entry(Gga, Gga, ws * (-Da/ha))   # row u1_a(v_j),  col u1_a(v_j)
                add_entry(Gob, Gga, ws * (-Db/hb))   # row u1_b,far,   col u1_a(v_j)
                add_entry(Ggb, Gga, ws * (+Db/hb))   # row u1_b(v_j),  col u1_a(v_j)

                # Trial fn u1_b (col = Ggb) — everything flips sign 
                add_entry(Goa, Ggb, ws * (-Da/ha))   # row u1_a,far,   col u1_b(v_j)
                add_entry(Gga, Ggb, ws * (+Da/ha))   # row u1_a(v_j),  col u1_b(v_j)
                add_entry(Gob, Ggb, ws * (+Db/hb))   # row u1_b,far,   col u1_b(v_j)
                add_entry(Ggb, Ggb, ws * (-Db/hb))   # row u1_b(v_j),  col u1_b(v_j)

    # finalize PETSc assembly
    M.assemblyBegin()
    M.assemblyEnd()

def advection_junction_terms_P1DG_flow_based(
    A_petsc, V1, Lambda, bif_vertices, D_area_expr, t_vec, velocity_expr
):
    dm = V1.dofmap()
    coords = Lambda.coordinates()
    Lambda.init(0, 1)
    v2c = Lambda.topology()(0, 1)

    N = A_petsc.mat().getSize()[0]
    off1 = N - V1.dim()

    M = A_petsc.mat()
    M.setOption(PETSc.Mat.Option.NEW_NONZERO_ALLOCATION_ERR, False)
    M.setOption(PETSc.Mat.Option.NEW_NONZERO_LOCATION_ERR, False)

    def add_entry(I, J, val):
        M.setValue(I, J, float(val), addv=True)

    _vel_arr = velocity_expr.vector().get_local()
    _dm_vel  = velocity_expr.function_space().dofmap()
    _n_proc  = 0
    _n_skip  = 0

    for vtx in bif_vertices:
        cells = list(v2c(vtx))
        if len(cells) < 3:
            continue

        xg = coords[vtx]
        branches = []

        for c in cells:
            cell = Cell(Lambda, c)
            vids = cell.entities(0)
            x0, x1 = coords[vids[0]], coords[vids[1]]

            # vector from junction -> other endpoint (points into the branch)
            if vids[0] == vtx:
                loc_g = 0
                xother = x1
            else:
                loc_g = 1
                xother = x0

            d = xother - xg
            d_hat = d / np.linalg.norm(d)

            x_mid = 0.5*(xg + xother)
            tmid  = np.array(t_vec(Point(*x_mid)))
            _tn   = np.linalg.norm(tmid)
            if _tn < 1e-14:
                continue
            tmid /= _tn

            # signed velocity along this branch away from junction
            v_mag = float(_vel_arr[int(_dm_vel.cell_dofs(c)[0])])
            s = v_mag * float(np.dot(tmid, d_hat))

            A = float(D_area_expr(Point(*x_mid)))
            Q = A * abs(s)

            dof_g = int(dm.cell_dofs(c)[loc_g])

            branches.append(dict(cell=c, dof_g=dof_g, s=s, Q=Q))

        inflows  = [b for b in branches if b["s"] < 0.0 and b["Q"] > 0.0]
        outflows = [b for b in branches if b["s"] > 0.0 and b["Q"] > 0.0]

        #print(f"Junction {vtx}: inflows={len(inflows)}, outflows={len(outflows)}, "
        #      f"s={[round(b['s'],4) for b in branches]}, "
        #      f"Q={[round(b['Q'],6) for b in branches]}")

        if len(outflows) == 0 or len(inflows) == 0:
            _n_skip += 1
            continue
        _n_proc += 1

        #   Gin (inflow junction dof) loses drug via junction outflow  -> +diagonal on Gin
        #   Gout (outflow junction dof) gains drug from Gin            -> -off-diagonal Gout<-Gin
        Q_in_sum = sum(b["Q"] for b in inflows)

        for bout in outflows:
            Gout = off1 + bout["dof_g"]
            Qout = bout["Q"]
            for bin_ in inflows:
                Gin = off1 + bin_["dof_g"]
                add_entry(Gout, Gin, Qout * (bin_["Q"] / Q_in_sum))
            add_entry(Gout, Gout, -Qout)

    print(f"[junc-adv] processed={_n_proc}, skipped={_n_skip} "
          f"(skipped means no clear inflow or all Q=0)")
    M.assemblyBegin()
    M.assemblyEnd()

def build_bfs_tangent_expression(Lambda, inlet_v, degree=0,return_dist=False):
    coords = Lambda.coordinates()
    conn   = Lambda.cells()  # (v0,v1) per cell

    # === BFS distances on the vertex graph ===
    adj = {v: [] for v in range(Lambda.num_vertices())}
    for a, b in conn:
        a = int(a); b = int(b)
        adj[a].append(b); adj[b].append(a)

    dist = {int(inlet_v): 0}
    q = deque([int(inlet_v)])
    while q:
        v = q.popleft()
        for nb in adj[v]:
            if nb not in dist:
                dist[nb] = dist[v] + 1
                q.append(nb)

    # === per-cell downstream tangent (from smaller dist -> larger dist) ===
    cell_mids = []
    cell_tans = []

    for c, (a, b) in enumerate(conn):
        a = int(a); b = int(b)
        da = dist.get(a, None)
        db = dist.get(b, None)
        if da is None or db is None:

            up, dn = a, b
        else:
            up, dn = (a, b) if da <= db else (b, a)

        x_up = coords[up]
        x_dn = coords[dn]
        t = x_dn - x_up
        t /= np.linalg.norm(t)

        cell_mids.append(0.5*(x_up + x_dn))
        cell_tans.append(t)

    cell_mids = np.array(cell_mids)
    cell_tans = np.array(cell_tans)
    kdt = cKDTree(cell_mids)

    class BFSTangent(UserExpression):
        def eval(self, values, x):
            _, idx = kdt.query((x[0], x[1], x[2]))
            values[:] = cell_tans[idx]
        def value_shape(self):
            return (3,)

    expr = BFSTangent(degree=degree)
    if return_dist:
      return expr, dist
    return expr

def build_vessel_tangent_extension_to_3d(Lambda, inlet_v, degree=0):
    """
    Returns:
      t3d : UserExpression on Omega that gives nearest-vessel tangent (unit vector)
      dist: BFS vertex distances (can reuse)
    """
    # 1) get per-cell downstream tangents and BFS dist
    t_cell, dist = build_bfs_tangent_expression(Lambda, inlet_v, degree=0, return_dist=True)  # you already have this

    # 2) sample Lambda cell midpoints and tangents there
    mids = []
    tans = []
    for c in range(Lambda.num_cells()):
        cell = df.Cell(Lambda, c)
        mp = cell.midpoint().array()
        mids.append(mp)
        tv = np.array(t_cell(df.Point(*mp)))
        nrm = np.linalg.norm(tv)
        if nrm < 1e-14:
            tv = np.array([0.0, 0.0, 1.0])
        else:
            tv = tv / nrm
        tans.append(tv)

    mids = np.asarray(mids)
    tans = np.asarray(tans)
    kdt  = cKDTree(mids)

    # 3) define nearest-neighbor extension into Omega
    class VesselTangent3D(df.UserExpression):
        def eval(self, values, x):
            _, idx = kdt.query((x[0], x[1], x[2]))
            values[0] = float(tans[idx, 0])
            values[1] = float(tans[idx, 1])
            values[2] = float(tans[idx, 2])
        def value_shape(self):
            return (3,)

    return VesselTangent3D(degree=degree)

def outlet_bn_geometric(Lambda, endpoints, Lambda_boundaries, t_vec, q1_value=1.0):
    coords = Lambda.coordinates()
    Lambda.init(0, 1)
    v2e = Lambda.topology()(0, 1)
    e2v = Lambda.topology()(1, 0)

    for v in endpoints:
        if Lambda_boundaries.array()[v] != 2:
            continue  # only outlets

        # endpoint has exactly one incident edge
        e = int(v2e(v)[0])
        vids = e2v(e)

        # other vertex on that edge
        v_other = int(vids[0] if vids[1] == v else vids[1])

        x_end   = coords[v]
        x_other = coords[v_other]

        # outward direction from interior -> endpoint
        d = x_end - x_other
        d_hat = d / np.linalg.norm(d)

        # evaluate your tangent (UserExpression) at edge midpoint
        x_mid = 0.5 * (x_end + x_other)
        tmid = np.array(t_vec(Point(*x_mid)))
        tmid = tmid / np.linalg.norm(tmid)

        bn = float(q1_value) * float(np.dot(tmid, d_hat))
        print(f"outlet vertex {v}: other={v_other}, bn≈{bn:+.6e}, tmid={tmid}, dout={d_hat}")

def classify_branch_cells(Lambda, bif_vtx):
    coords = Lambda.coordinates()
    bif_x = coords[bif_vtx]              # numpy array shape (3,)

    left = []
    right = []

    for c in range(Lambda.num_cells()):
        cell = Cell(Lambda, c)
        mpP = cell.midpoint()            # dolfin Point
        mp  = np.array(mpP.array())      # numpy array (3,)

        d = mp - bif_x                   # numpy - numpy works

        # ignore parent branch (z <= 0 relative to bif)
        if d[2] <= 0.0:
            continue

        # classify by x sign
        if mp[0] > 0.0:
            right.append((np.linalg.norm(d), c))
        else:
            left.append((np.linalg.norm(d), c))

    left.sort()
    right.sort()

    return [c for _, c in left], [c for _, c in right]

def compute_vertex_bfs_dist(Lambda, inlet_v):
    conn = Lambda.cells()
    adj = {v: [] for v in range(Lambda.num_vertices())}
    for a, b in conn:
        a = int(a); b = int(b)
        adj[a].append(b); adj[b].append(a)

    dist = {int(inlet_v): 0}
    q = deque([int(inlet_v)])
    while q:
        v = q.popleft()
        for nb in adj[v]:
            if nb not in dist:
                dist[nb] = dist[v] + 1
                q.append(nb)
    return dist

def cube_around_Lambda(Lambda, pad_factor, ncell):
    """
    Returns:
      mesh_3d : BoxMesh (cube)
      lo, hi  : cube corners (numpy arrays)
    pad_factor: fraction of cube side length added as padding on each side.
    """
    X = Lambda.coordinates()
    xyz_min = X.min(axis=0)
    xyz_max = X.max(axis=0)

    center = 0.5*(xyz_min + xyz_max)
    spans  = xyz_max - xyz_min
    L = float(spans.max())          # cube side length BEFORE padding
    pad = pad_factor * L            # padding each side

    half = 0.5*L + pad
    lo = center - half
    hi = center + half

    mesh_3d = BoxMesh(Point(*lo), Point(*hi), ncell, ncell, ncell)
    return mesh_3d, lo, hi

# May switch back to this function for general network
class radius_function(UserExpression):
    def __init__(self, G, node_list, kdtree, **kwargs):
        self.G = G
        self.node_list = node_list
        self.kdtree = kdtree
        super().__init__(**kwargs)
    def eval(self, value, x):
        _, idx = self.kdtree.query((x[0], x[1], x[2]))
        node = self.node_list[idx]
        value[0] = float(self.G.nodes[node]['radius'])
    def value_shape(self):
        return ()

# Currently using this function due to how mesh was created
class edge_radius_function(UserExpression):
    def __init__(self, edge_midpoints, edge_radii, kdtree, **kwargs):
        super().__init__(**kwargs)
        self.edge_midpoints = edge_midpoints
        self.edge_radii = edge_radii
        self.kdtree = kdtree

    def eval(self, value, x):
        _, idx = self.kdtree.query((x[0], x[1], x[2]))
        value[0] = float(self.edge_radii[idx])

    def value_shape(self):
        return ()

def build_radius_map_constant_per_vessel(G, vessel_radius, degree=0):
    """
    Build a radius_map_G that is constant per vessel_id (chain),
    using nearest-edge-midpoint lookup.
    """
    pos = nx.get_node_attributes(G, "pos")

    edge_midpoints = []
    edge_radii = []

    for (a, b) in G.edges():
        pa = np.array(pos[a], dtype=float)
        pb = np.array(pos[b], dtype=float)
        pmid = 0.5 * (pa + pb)

        vid = G.edges[a, b].get("vessel_id", None)
        if vid is None:
            raise RuntimeError("Missing vessel_id on some edges. Run assign_vessel_ids_by_chains(G) first.")

        r = float(vessel_radius[int(vid)])
        edge_midpoints.append(pmid)
        edge_radii.append(r)

    edge_midpoints = np.asarray(edge_midpoints, dtype=float)
    edge_radii = np.asarray(edge_radii, dtype=float)

    kdt = cKDTree(edge_midpoints)
    radius_map_G = edge_radius_function(edge_midpoints, edge_radii, kdt, degree=degree)

    return radius_map_G, edge_midpoints, edge_radii

# L_max computed in Cell 2 after Y-bif graph is built

def make_Y_bifurcation(
    parent_len=0.45, parent_pts=4,          # parent vessel
    daughter_len=0.45, daughter_pts=4,      # identical daughter vessels
    angle_deg=35.0,                         # opening half-angle of the Y (each daughter)
    r_parent=0.05,
    r_daughter1=0.040,
    r_daughter2=0.020,
    damage_default=0.0,                     # damage field included in vtk file from Pierce's example, just made zero
    origin=(0.0, 0.0, -0.45),               # parent begins at z = -0.45
    bifurcation_z= 0.0                      # split occors at origin
):
    G = FenicsGraph()

    # === Parent: from (0,0,-0.45) to (0,0,bifurcation_z) ===
    z0 = origin[2]
    parent_zs = np.linspace(z0, bifurcation_z, parent_pts)
    parent_ids = []
    for z in parent_zs:
        i = len(G)  # next node id
        G.add_node(i, pos=(origin[0], origin[1], z), radius=r_parent, damage=damage_default)
        parent_ids.append(i)

    # Use the last parent node as the bifurcation node:
    bif_id = parent_ids[-1]
    bif_pt = G.nodes[bif_id]["pos"]

    # Connect parent and daughter paths
    for a, b in zip(parent_ids[:-1], parent_ids[1:]):
        G.add_edge(a, b)

    # === Daughters: symmetric about the z-axis in x–z plane ===
    angle = np.deg2rad(angle_deg)
    # Unit directions for the two daughters (right and left in +x/-x):
    d1 = np.array([ np.sin(angle), 0.0, np.cos(angle)])
    d2 = np.array([-np.sin(angle), 0.0, np.cos(angle)])

    def add_branch(bif_id, dir_vec, r_daughter):
        # start at the bifurcation point (re-use the node id at the bifurcation)
        # then create additional points along the ray
        ids = [bif_id]
        for k in range(1, daughter_pts):
            p = np.array(bif_pt) + (k / (daughter_pts - 1)) * daughter_len * dir_vec
            i = len(G)
            G.add_node(i, pos=tuple(p), radius=r_daughter, damage=damage_default)
            ids.append(i)
        # connect the line
        for a, b in zip(ids[:-1], ids[1:]):
            G.add_edge(a, b)

    add_branch(bif_id, d1, r_daughter1)
    add_branch(bif_id, d2, r_daughter2)

    return G

def kirchhoff_residual_at_vertex(Lambda, V1, uh1d, vtx, D_area_expr):
  coords = Lambda.coordinates()
  dm = V1.dofmap()

  Lambda.init(0, 1)
  v2c = Lambda.topology()(0, 1)
  cells = list(v2c(vtx))

  res = 0.0
  fluxes = []
  for c in cells:
      cell = Cell(Lambda, c)
      vids = cell.entities(0)
      x0 = coords[vids[0]]
      x1 = coords[vids[1]]
      hK = float(np.linalg.norm(x1 - x0))
      x_mid = 0.5*(x0 + x1)
      Da = float(D_area_expr(Point(*x_mid)))

      cdofs = dm.cell_dofs(c)
      # local dof at junction vs other endpoint
      if vids[0] == vtx:
          dof_g, dof_o = int(cdofs[0]), int(cdofs[1])
      else:
          dof_g, dof_o = int(cdofs[1]), int(cdofs[0])

      ug = uh1d.vector()[dof_g]
      uo = uh1d.vector()[dof_o]
      F = Da * (uo - ug) / hK  # outgoing flux from junction into cell
      fluxes.append(F)
      res += F

  return res, fluxes


/opt/conda/lib/python3.8/site-packages/block/__init__.py:15: UserWarning: The cbc.block repository has moved to https://github.com/blocknics/cbc.block
  warnings.warn('The cbc.block repository has moved to https://github.com/blocknics/cbc.block', UserWarning)
Missing HsMG for fract norm computing


In [2]:
#@title Y-bifurcation graph setup

# Build a Y-bifurcation with asymmetric daughter radii for testing.
# Parent:      r = 0.05 mm   (along z-axis, inlet at bottom)
# Daughter 1:  r = 0.04 mm   (right branch, +x direction)
# Daughter 2:  r = 0.02 mm   (left branch,  -x direction)
# Asymmetric radii give unequal flow splits — a good test that the
# pressure solve, velocity, and advection are all physically consistent.

G = make_Y_bifurcation(
    parent_len=0.45, parent_pts=4,
    daughter_len=0.45, daughter_pts=4,
    angle_deg=35.0,
    r_parent=0.05,
    r_daughter1=0.04,
    r_daughter2=0.02,
)

nv = assign_vessel_ids_by_junction_terminal_paths(G, min_edge_length=1e-6)

junctions = [n for n in G.nodes if G.degree[n] > 2]
terminals  = [n for n in G.nodes if G.degree[n] == 1]
print(f"#nodes: {len(G.nodes)}, #edges: {len(G.edges)}")
print(f"#junctions: {len(junctions)}, #terminals: {len(terminals)}")
print(f"Junction nodes: {junctions}")
print(f"Terminal nodes: {terminals}")

# Identify root = inlet = lowest-z terminal
pos = dict(G.nodes(data="pos"))
root = min(terminals, key=lambda n: pos[n][2])
print(f"Root (inlet): vertex {root}, pos={pos[root]}")

# Compute longest inlet->terminal path for transit time estimate in Cell 3
pos_g = nx.get_node_attributes(G, 'pos')
for u, v in G.edges():
    G.edges[u, v]['length'] = float(np.linalg.norm(np.array(pos_g[u]) - np.array(pos_g[v])))
path_lengths = nx.single_source_dijkstra_path_length(G, root, weight='length')
L_max = max(path_lengths.values())
print(f"Longest inlet->terminal path: {L_max:.3f} mm")


#nodes: 10, #edges: 9
#junctions: 1, #terminals: 3
Junction nodes: [3]
Terminal nodes: [0, 6, 9]
Root (inlet): vertex 0, pos=(0.0, 0.0, -0.45)
Longest inlet->terminal path: 0.900 mm


In [3]:
#@title Y-bifurcation simulation

# ============================================================
# === Parameters ===
# ============================================================
# Simulation time
T      = 0.5     
dt     = 0.001  
inv_dt = Constant(1.0 / dt)
tpts   = int(T / dt)
tau    = 0.20    # duration of drug administration (s)

# Drug / concentration
cinlet    = Constant(10.0)  # drug concentration at inlet
Diff_coef = Constant(1.0)   # mm^2/s, diffusion coefficient
kappa_val  = 0.1             # 1D-3D exchange coefficient
d_exchange = 10.0            # mm — exchange active within this distance of outlet terminals

# Mesh refinement
ncell = 16  

# ============================================================
# === Mesh Construction ===
# ============================================================
import numpy as np

G.make_mesh()
Lambda, mf_Lambda = G.get_mesh()
nodes, P_nodes, T_nodes = bif_tangents(G)
mesh_3d, lo, hi = cube_around_Lambda(Lambda, pad_factor=0.05, ncell=ncell)

coords_check = Lambda.coordinates()
print(f"1D mesh: {Lambda.num_cells()} cells, h_max={Lambda.hmax():.5f}")

# ============================================================
# === 3D Box Boundary Markers ===
# ============================================================
Omega_boundaries = MeshFunction("size_t", mesh_3d, mesh_3d.topology().dim() - 1, 0)
eps = 1e-10
x0,y0,z0 = lo
x1,y1,z1 = hi
CompiledSubDomain("near(x[0], x0, tol)", x0=x0, tol=eps).mark(Omega_boundaries, 1)
CompiledSubDomain("near(x[0], x1, tol)", x1=x1, tol=eps).mark(Omega_boundaries, 2)
CompiledSubDomain("near(x[1], y0, tol)", y0=y0, tol=eps).mark(Omega_boundaries, 3)
CompiledSubDomain("near(x[1], y1, tol)", y1=y1, tol=eps).mark(Omega_boundaries, 4)
CompiledSubDomain("near(x[2], z0, tol)", z0=z0, tol=eps).mark(Omega_boundaries, 5)
CompiledSubDomain("near(x[2], z1, tol)", z1=z1, tol=eps).mark(Omega_boundaries, 6)

# ============================================================
# === 1D Network Boundary Markers ===
# ============================================================
Lambda_boundaries = MeshFunction("size_t", Lambda, 0, 0)
coords = Lambda.coordinates()

Lambda.init(0,1); v2e = Lambda.topology()(0,1)
endpoints = [v for v in range(Lambda.num_vertices()) if v2e.size(v)==1]
inlet_point = np.array(G.nodes[root]["pos"], dtype=float)
diff_v = coords - inlet_point
inlet_v = int(np.argmin(np.linalg.norm(diff_v, axis=1)))

Lambda_boundaries.array()[:] = 0
Lambda_boundaries.array()[inlet_v] = 1
for v in endpoints:
    if v != inlet_v:
        Lambda_boundaries.array()[v] = 2
print(f"inlet_v={inlet_v}, outlets={[v for v in endpoints if v != inlet_v]}")

# ============================================================
# === Inlet Facet Detection ===
# ============================================================
# Detect whether the inlet is a topological leaf (exterior) or an interior vertex.
# For this Y bifurcation the inlet is always a leaf.
inlet_is_leaf = inlet_v in endpoints
if not inlet_is_leaf:
    inlet_tag = MeshFunction("size_t", Lambda, 0, 0)
    inlet_tag.array()[inlet_v] = 1
    dS_inlet = Measure("dS", domain=Lambda, subdomain_data=inlet_tag)(1)

# ============================================================
# === Vessel Geometry ===
# ============================================================
t_vec, dist1d = build_bfs_tangent_expression(Lambda, inlet_v, degree=0, return_dist=True)
t_vec3d = build_vessel_tangent_extension_to_3d(Lambda, inlet_v, degree=0)
outlet_bn_geometric(Lambda, endpoints, Lambda_boundaries, t_vec, q1_value=5.0)

# Vessel radii — read from graph nodes (set in make_Y_bifurcation)
# Build vessel_radius from edges: assign_vessel_ids sets vessel_id on edges,
# and make_Y_bifurcation stores radius on each node.
vessel_radius = {}
for u, v, edata in G.edges(data=True):
    vid = edata.get("vessel_id", None)
    if vid is not None:
        r = 0.5 * (G.nodes[u].get("radius", 0.05) + G.nodes[v].get("radius", 0.05))
        vessel_radius.setdefault(int(vid), float(r))
print(f"vessel_radius: {vessel_radius}")

radius_map_G, edge_midpoints, edge_radii = build_radius_map_constant_per_vessel(
    G, vessel_radius, degree=0
)

# === Perimeter and Area dependent on position ===
D_area       = np.pi * radius_map_G**2
D_perimeter  = 2.0  * np.pi * radius_map_G
cylinder     = Circle(radius=radius_map_G, degree=10)

# ============================================================
# === Bifurcation Topology ===
# ============================================================
bif_info = build_bifurcation_pairs_bifs(G, root, require_tree_like=True)
bif_vertices = bifurcation_vertices_from_graph(Lambda, bif_info)
outlet_vs = [v for v in endpoints if Lambda_boundaries.array()[v] == 2]
print(f"bif_vertices: {bif_vertices}")
print(f"outlet_vs:    {outlet_vs}")

# === Split dS measure for junction and non-junction ===
facet_tags = MeshFunction("size_t", Lambda, 0, 0)
for vtx in bif_vertices:              # tag junction vertices so dS_reg excludes them --
    facet_tags.array()[vtx] = 1       # dS at a 3-cell vertex is non-manifold/undefined
# Junction facets are excluded from dS_reg (tagged=1) so FEniCS dS does not attempt
# pairwise coupling at non-manifold trifurcation vertices. 

# ============================================================
# === Function Spaces & Initial Conditions ===
# ============================================================
V3     = FunctionSpace(mesh_3d, "DG", 1)
V1     = FunctionSpace(Lambda, "DG", 1)
W      = [V3, V1]
u3, u1 = map(TrialFunction, W)
v3, v1 = map(TestFunction, W)

uh3d_prev = interpolate(Expression("0.0", degree=1), V3)
uh1d_prev = interpolate(Expression("0.0", degree=1), V1)

pulse = Constant(0.0)  # set to 1.0 during injection window [0, tau]

# ============================================================
# === Mesh Measures & Stabilization Parameters ===
# ============================================================
dxOmega  = Measure("dx", domain=mesh_3d)
dsOmega  = Measure("ds", domain=mesh_3d, subdomain_data=Omega_boundaries)
dSOmega  = Measure("dS", domain=mesh_3d)
dxLambda = Measure("dx", domain=Lambda)
dsLambda = Measure("ds", domain=Lambda, subdomain_data=Lambda_boundaries)
dSLambda = Measure("dS", domain=Lambda)
dS_tag   = Measure("dS", domain=Lambda, subdomain_data=facet_tags)
dS_jun   = dS_tag(1)
dS_reg   = dS_tag(0)
n3d      = FacetNormal(mesh_3d)
n1d      = FacetNormal(Lambda)
h3d      = CellDiameter(mesh_3d)
h1d      = CellDiameter(Lambda)
alpha3d  = Constant(50.0)
alpha1d  = Constant(50.0)
epsilon  = Constant(1.0)
g        = Constant(0.0)

# ============================================================
# === Pressure & Velocity Solve ===
# ============================================================
# Solve:  div(kappa_press * grad(p)) = 0  with p in mmHg
# Velocity [mm/s] = -(r^2/8mu) * dp/ds  (Hagen-Poiseuille, Darcy along tangent)
MU_BLOOD_MMHG_S = 3.5e-3 / 133.322          # blood viscosity: 3.5e-3 Pa*s -> mmHg*s
kappa_press = (np.pi / (8.0 * MU_BLOOD_MMHG_S)) * radius_map_G**4
kappa_vel   = radius_map_G**2 / (8.0 * MU_BLOOD_MMHG_S)

P_INLET = 6.0     # inlet pressure (mmHg) — dp=1 mmHg → v~13 mm/s, Pe~0.65
P_OUT   = 5.0     # outlet pressure (mmHg)

V_p   = FunctionSpace(Lambda, "CG", 1)
p_h   = Function(V_p, name="pressure")
p_trl = TrialFunction(V_p)
w_p   = TestFunction(V_p)

a_press = kappa_press * dot(grad(p_trl), grad(w_p)) * dxLambda
L_press = Constant(0.0) * w_p * dxLambda

bc_p_in  = DirichletBC(V_p, Constant(P_INLET), Lambda_boundaries, 1)
bc_p_out = DirichletBC(V_p, Constant(P_OUT),   Lambda_boundaries, 2)
solve(a_press == L_press, p_h, [bc_p_in, bc_p_out])
print(f"Pressure: p in [{p_h.vector().min():.4f}, {p_h.vector().max():.4f}] mmHg")

V1_DG0      = FunctionSpace(Lambda, "DG", 0)
q_scalar_fn = project(-kappa_vel * dot(grad(p_h), t_vec), V1_DG0)
q_arr = q_scalar_fn.vector().get_local()
q_arr[q_arr < 0.0] = 0.0          # clip numerical negatives in stagnant branches
q_scalar_fn.vector().set_local(q_arr)
q_scalar_fn.vector().apply("insert")
print(f"Velocity [mm/s]: v in [{q_scalar_fn.vector().min():.4f}, {q_scalar_fn.vector().max():.4f}]")
_q_max_phys = float(q_scalar_fn.vector().max())
if _q_max_phys > 0:
    T_transit = L_max / _q_max_phys
    print(f"Estimated transit time: {T_transit:.2f} s (L_max={L_max:.3f} mm, v_max={_q_max_phys:.4f} mm/s)")

File("ybif_pressure.pvd") << p_h
File("ybif_velocity.pvd") << q_scalar_fn

# ============================================================
# === Exchange Field ===
# ============================================================
# Spatially-varying exchange: leaky only near outlet terminals.
outlet_coords = np.array([Lambda.coordinates()[v] for v in outlet_vs])
coords_lam    = Lambda.coordinates()
mask_vals     = np.zeros(Lambda.num_cells())
outlet_set    = set(outlet_vs)
for _c in cells(Lambda):
    vids  = _c.entities(0)
    if vids[0] in outlet_set or vids[1] in outlet_set:
        continue
    midpt = 0.5 * (coords_lam[vids[0]] + coords_lam[vids[1]])
    if np.linalg.norm(outlet_coords - midpt[None, :], axis=1).min() < d_exchange:
        mask_vals[_c.index()] = 1.0

exchange_mask_fn = Function(V1_DG0, name="exchange_mask")
exchange_mask_fn.vector().set_local(mask_vals)
exchange_mask_fn.vector().apply("insert")
print(f"Exchange active on {int(mask_vals.sum())}/{Lambda.num_cells()} cells (d_exchange={d_exchange})")

_mask_vals = mask_vals.copy()
class ExchangeMask(UserExpression):
    def eval_cell(self, values, x, cell):
        values[0] = float(_mask_vals[cell.index])
    def value_shape(self):
        return ()
kappa = kappa_val * ExchangeMask(degree=0)

# ============================================================
# === Variational Form Parameters ===
# ============================================================
q1     = q_scalar_fn    # Darcy velocity from pressure solve
q_t    = q1 * t_vec
q3     = as_vector((0,0,0))
q3_pos = dot(q3('+'), n3d('+'))

bn1 = dot(q_t, n1d)
u1_bdry_up = conditional(ge(bn1, 0.0), u1, u1*0)

bn3 = dot(q3, n3d)
u3_bdry_up = conditional(ge(bn3, 0.0), u3, u3*0)

# === Upwind ===
q_norm_1d = dot(q_t('+'), n1d('+'))
u1_up = conditional(ge(q_norm_1d, 0.0), u1('+'), u1('-'))
u3_up = conditional(ge(q3_pos,    0.0), u3('+'), u3('-'))

# === Averages ===
u3_avg = Average(u3, Lambda, cylinder)
v3_avg = Average(v3, Lambda, cylinder)

# ============================================================
# === Time Loop Setup ===
# ============================================================
save_steps = {0}
for k in range(1, 9):
    save_steps.add(round(tpts * k / 8))
save_steps.add(tpts - 1)
print(f"Saving at steps: {sorted(save_steps)}")

vtu3 = File("ybif_3d_series.pvd")
vtu1 = File("ybif_1d_series.pvd")

# ============================================================
# === Time Stepping ===
# ============================================================
t = 0.0
for step in range(tpts):
    t += dt
    pulse.assign(1.0 if t <= tau else 0.0)

    #cinlet.t = t

    # === Variational Forms ===
    a00 = inv_dt * (inner(u3, v3) * dxOmega) \
          + Diff_coef * inner(grad(u3), grad(v3)) * dxOmega \
          + D_perimeter * kappa * inner(u3_avg, v3_avg) * dxLambda \
          - Diff_coef * epsilon * dot(avg(grad(v3)),jump(u3,n3d))* dSOmega \
          - Diff_coef * dot(jump(v3,n3d),avg(grad(u3))) * dSOmega \
          + (alpha3d/avg(h3d)) * dot(jump(v3,n3d),jump(u3,n3d)) * dSOmega \
          - inner(q3, grad(v3)) * u3 * dxOmega \
          + q3_pos * u3_up * jump(v3) * dSOmega \
          + bn3 * u3_bdry_up * v3 * (dsOmega(1) + dsOmega(2) + dsOmega(3) + dsOmega(4) + dsOmega(5) + dsOmega(6))
    a01 = -kappa * D_perimeter * inner(u1, v3_avg) * dxLambda
    a10 = -kappa * D_perimeter * inner(u3_avg, v1) * dxLambda
    a11 = D_area * inv_dt * u1 * v1 * dxLambda \
          + Diff_coef * D_area * inner(grad(u1), grad(v1)) * dxLambda \
          + (alpha1d * avg(D_area) / avg(h1d)) * jump(u1) * jump(v1) * dS_reg \
          - Diff_coef * avg(D_area) * inner(avg(grad(u1)), jump(v1, n1d)) * dS_reg \
          - Diff_coef * epsilon * avg(D_area) * inner(avg(grad(v1)), jump(u1, n1d)) * dS_reg \
          + kappa * D_perimeter * u1 * v1 * dxLambda \
          - D_area * u1 * dot(q_t, grad(v1)) * dxLambda \
          + avg(D_area) * q_norm_1d * u1_up * jump(v1) * dS_reg \
          + D_area * bn1 * u1_bdry_up * v1 * dsLambda(2)  # add + dsLambda(3) if targeting tumor

    f3 = Constant(0.0)
    f1 = Constant(0.0)

    L0 = inner(f3, v3) * dxOmega + inv_dt * inner(uh3d_prev, v3) * dxOmega

    if inlet_is_leaf:
        inflow_term = pulse * D_area * q1 * cinlet * v1 * dsLambda(1)
    else:
        inflow_term = (pulse * D_area("+") * q1("+") * cinlet * v1("+") * dS_inlet
                      + pulse * D_area("-") * q1("-") * cinlet * v1("-") * dS_inlet)

    L1 = inner(f1, v1) * dxLambda + D_area * inv_dt * uh1d_prev * v1 * dxLambda \
        + inflow_term

    a = [[a00, a01], [a10, a11]]
    L = [L0, L1]

    W_bcs = [[], []]  # Boundaries included in variational form for dG

    A, b = map(ii_assemble, (a, L))
    A, b = apply_bc(A, b, W_bcs)
    A, b = map(ii_convert, (A, b))

    # === Kirchhoff junction terms (diffusion, DG1) ===
    add_kirchhoff_junction_terms_P1DG(
        A, V3, V1, Lambda, bif_vertices,
        D_area_expr=D_area,
        eta_gamma=50.0,
        epsilon=1.0,
        diff_coef=float(Diff_coef)
    )
    A.apply("insert")

    # Advection junction (enabled):
    advection_junction_terms_P1DG_flow_based(A, V1, Lambda, bif_vertices, D_area, t_vec, q_scalar_fn)
    A.apply("insert")

    solver = PETScKrylovSolver()
    solver.set_operators(A, A)
    ksp = solver.ksp()
    opts = PETSc.Options()
    opts.setValue("ksp_type", "preonly")
    opts.setValue("pc_type", "lu")
    opts.setValue("pc_factor_mat_solver_type", "mumps")
    opts.setValue("ksp_atol", 1E-14)
    opts.setValue("ksp_rtol", 1E-8)
    ksp.setFromOptions()

    x = b.copy()
    solver.solve(A, x, b)

    wh = ii_Function(W)
    wh.vector()[:] = x
    uh3d, uh1d = wh

    uh3d_prev.assign(uh3d)
    uh1d_prev.assign(uh1d)

    u1min = uh1d.vector().min()
    u1max = uh1d.vector().max()
    if step % 50 == 0 or step == tpts - 1:
        print(f"[step {step:4d}] t={t:.3f}  u1=[{u1min:.3e}, {u1max:.3e}]")


    # Kirchhoff diagnostic at key steps (front crosses junction at ~step 35 for these params)
    if step == 30:
        print(f"\n  --- Kirchhoff flux residual [step {step}, t={t:.4f} s] ---")
        for vtx in bif_vertices:
            res, fluxes = kirchhoff_residual_at_vertex(Lambda, V1, uh1d, vtx, D_area)
            max_f = max(abs(f) for f in fluxes) if fluxes else 1.0
            ratio = abs(res) / max_f if max_f > 1e-30 else float('nan')
            print(f"  Vertex {vtx}: residual={res:.3e}, max|flux|={max_f:.3e}, |res|/max={ratio:.4f}")
            print(f"  Branch fluxes: {[f'{f:.3e}' for f in fluxes]}")
        print()
        uh1d_step30 = uh1d.copy(deepcopy=True)

    if step in save_steps:
        uh3d.rename("u3", "u3"); vtu3 << (uh3d, t)
        uh1d.rename("u1", "u1"); vtu1 << (uh1d, t)

# === Final time-step only ===
File("ybif_3d_final.pvd") << uh3d
File("ybif_1d_final.pvd") << uh1d
print("Simulation complete.")


Averaging over 18 cells: 100%|##########| 18/18 [00:00<00:00, 1320.14it/s]
1D mesh: 18 cells, h_max=0.07500
inlet_v=0, outlets=[6, 9]
outlet vertex 6: other=16, bn≈+5.000000e+00, tmid=[0.57357644 0.         0.81915204], dout=[0.57357644 0.         0.81915204]
outlet vertex 9: other=18, bn≈+5.000000e+00, tmid=[-0.57357644  0.          0.81915204], dout=[-0.57357644  0.          0.81915204]
vessel_radius: {0: 0.05, 1: 0.045, 2: 0.035}
bif_vertices: [3]
outlet_vs:    [6, 9]
Pressure: p in [5.0000, 6.0000] mmHg
Velocity [mm/s]: v in [6.8357, 12.5024]
Estimated transit time: 0.07 s (L_max=0.900 mm, v_max=12.5024 mm/s)
Exchange active on 16/18 cells (d_exchange=10.0)
Saving at steps: [0, 62, 125, 188, 250, 312, 375, 438, 499, 500]
[junc-adv] processed=1, skipped=0 (skipped means no clear inflow or all Q=0)
[step    0] t=0.001  u1=[1.250e-14, 2.770e+00]
[junc-adv] processed=1, skipped=0 (skipped means no clear inflow or all Q=0)
[junc-adv] processed=1, skipped=0 (skipped means no clear inflow

In [4]:
#@title Kirchhoff junction diagnostics

# ============================================================
# === 1. Junction topology ===
# ============================================================
Lambda.init(0, 1)
v2c = Lambda.topology()(0, 1)
print("=== Junction vertex topology ===")
for vtx in bif_vertices:
    incident = list(v2c(vtx))
    vtx_coord = Lambda.coordinates()[vtx]
    print(f"  Junction vertex {vtx} at {vtx_coord.tolist()}")
    print(f"  Incident cells: {incident}")
    for c in incident:
        cell = Cell(Lambda, c)
        vids = list(cell.entities(0))
        other = [v for v in vids if v != vtx][0]
        r_mid = float(radius_map_G(Point(*cell.midpoint().array())))
        print(f"    Cell {c}: vertices={vids}, other_vtx={other}, r={r_mid:.4f} mm")

# ============================================================
# === 2. Kirchhoff flux residual (diffusion) ===
# ============================================================
# At each junction vertex, the sum of diffusive fluxes D*A*(u_other - u_junc)/h
# across all incident branches should be zero (flux balance / Kirchhoff condition).
# A small residual confirms the junction terms are enforcing continuity of flux.
print("\n=== Kirchhoff diffusion flux residual (step 30, t=0.031 s) ===")
for vtx in bif_vertices:
    res, fluxes = kirchhoff_residual_at_vertex(Lambda, V1, uh1d_step30, vtx, D_area)
    max_f = max(abs(f) for f in fluxes) if fluxes else 1.0
    ratio = abs(res) / max_f if max_f > 1e-30 else float('nan')
    print(f"  Vertex {vtx}: residual={res:.3e}, max|flux|={max_f:.3e}, |res|/max={ratio:.4f}")
    print(f"  Branch fluxes: {[f'{f:.3e}' for f in fluxes]}")
    print(f"  (|res|/max << 1 is good; front crosses junction at step ~30-35 for these params)")

# ============================================================
# === 3. Concentration values at junction and daughters ===
# ============================================================
# Sample concentration at the junction vertex and at the midpoint of each daughter branch.
print("\n=== Concentration at junction and daughter branches (final step) ===")
for vtx in bif_vertices:
    incident = list(v2c(vtx))
    vtx_pt = Point(*Lambda.coordinates()[vtx])
    u_junc = float(uh1d(vtx_pt))
    print(f"  Junction vertex {vtx}: u1 = {u_junc:.6e}")
    for c in incident:
        cell = Cell(Lambda, c)
        vids = list(cell.entities(0))
        other = [v for v in vids if v != vtx][0]
        # classify: parent (upstream) or daughter (downstream)
        d_vtx   = dist1d.get(vtx, 0.0)
        d_other = dist1d.get(other, 0.0)
        branch_type = "parent" if d_other < d_vtx else "daughter"
        r_mid = float(radius_map_G(Point(*cell.midpoint().array())))
        u_mid = float(uh1d(cell.midpoint()))
        print(f"    Cell {c} ({branch_type}, r={r_mid:.4f} mm): u1_mid = {u_mid:.6e}")

# ============================================================
# === 4. Mass conservation check ===
# ============================================================
# Total drug mass in the 1D network: integral of u1 * D_area over Lambda.
# After injection stops (t > tau), mass should decrease monotonically.
print("\n=== Mass in 1D network at final step ===")
mass_1d = assemble(uh1d * D_area * dxLambda)
print(f"  integral(u1 * D_area) over Lambda = {mass_1d:.6e}")

# ============================================================
# === 5. Velocity split at junction ===
# ============================================================
# For a physical flow, the flux (velocity * cross-section) into the junction
# from the parent should equal the sum of fluxes out through the daughters.
# Checks that the pressure solve produced a physically consistent flow split.
print("\n=== Flow flux balance at junction (velocity check) ===")
for vtx in bif_vertices:
    incident = list(v2c(vtx))
    d_vtx = dist1d.get(vtx, 0.0)
    net_flux = 0.0
    for c in incident:
        cell = Cell(Lambda, c)
        vids = list(cell.entities(0))
        other = [v for v in vids if v != vtx][0]
        d_other = dist1d.get(other, 0.0)
        # sign: +1 if flow goes away from junction (outward), -1 if into junction
        sign = +1.0 if d_other > d_vtx else -1.0
        r_mid   = float(radius_map_G(Point(*cell.midpoint().array())))
        A_mid   = np.pi * r_mid**2
        q_mid   = float(q_scalar_fn(cell.midpoint()))
        branch_type = "parent" if d_other < d_vtx else "daughter"
        print(f"    Cell {c} ({branch_type}): q={q_mid:.4f} mm/s, A={A_mid:.6f} mm^2, Q=q*A={q_mid*A_mid:.6e}")
        net_flux += sign * q_mid * A_mid
    print(f"  Net flux imbalance at vertex {vtx}: {net_flux:.6e}  (should be ~0)")


=== Junction vertex topology ===
  Junction vertex 3 at [0.0, 0.0, 0.0]
  Incident cells: [5, 6, 8]
    Cell 5: vertices=[3, 12], other_vtx=12, r=0.0500 mm
    Cell 6: vertices=[3, 13], other_vtx=13, r=0.0450 mm
    Cell 8: vertices=[3, 14], other_vtx=14, r=0.0350 mm

=== Kirchhoff diffusion flux residual (step 30, t=0.031 s) ===
  Vertex 3: residual=7.722e-03, max|flux|=1.466e-01, |res|/max=0.0527
  Branch fluxes: ['1.466e-01', '-7.985e-02', '-5.901e-02']
  (|res|/max << 1 is good; front crosses junction at step ~30-35 for these params)

=== Concentration at junction and daughter branches (final step) ===
  Junction vertex 3: u1 = 1.281478e-02
    Cell 5 (parent, r=0.0500 mm): u1_mid = 1.149681e-02
    Cell 6 (daughter, r=0.0450 mm): u1_mid = 1.358088e-02
    Cell 8 (daughter, r=0.0350 mm): u1_mid = 1.452146e-02

=== Mass in 1D network at final step ===
  integral(u1 * D_area) over Lambda = 1.104588e-04

=== Flow flux balance at junction (velocity check) ===
    Cell 5 (parent): q=12.